In [ ]:
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_chroma import Chroma
from langgraph.graph import StateGraph, START, END
from langchain_core.documents import Document
from dotenv import load_dotenv
from typing import TypedDict, Literal, List
from langchain_text_splitters import RecursiveCharacterTextSplitter
import os

load_dotenv()

In [ ]:
# 1. Load documents

DOCUMENT_PATH = "./knowledge_base"

def load_documents(folder_path: str) -> List[Document]:
    documents = []

    for filename in os.listdir(folder_path):
        # print(filename)
        if filename.endswith(".txt"):
            file_path = os.path.join(folder_path, filename)
            # print(file_path)

            with open(file_path, 'r', encoding="utf-8") as file:
                text = file.read()
                doc = Document(
                    page_content=text,
                    metadata={
                        "source": filename
                    }
                )

                documents.append(doc)

    return documents


raw_documents = load_documents(DOCUMENT_PATH)
print(raw_documents)

In [ ]:
# 2. Chunk all the documents

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)

chunks = text_splitter.split_documents(raw_documents)
print(f"Number of chunks: {len(chunks)}")
print(f"Chunks: {chunks}")


In [ ]:
# 3. Create embeddings and vector store

embeddings = OpenAIEmbeddings(
    model="text-embedding-3-small"
)

vectore_store = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings
)

# Get our retriever to be used later
retriever = vectore_store.as_retriever(
    search_kwargs={"k": 5},
)

In [ ]:
# 4. Set up our LLM
llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0
)

In [ ]:
# 5. Define LangGraph state
class AgentState(TypedDict):
    question: str
    route: str
    documents: List[Document]
    answer: str
    review: str
    final_answer: str


In [ ]:
# 6. Create a routing node

def question_router_node(state: AgentState) -> dict:
    question = state['question']

    prompt = f"""
You are a routing assistant.

Decide whether the user question requires university knowledge documents.

Route as:
- "knowledge" if the question is about university policies, tuition, housing, registration, financial aid, student services, international students, career services, GPA, attendance, or campus support.
- "general" if the question is a general question unrelated to the university.

Question:
{question}

Return only one word: knowledge or general.
"""

    response = llm.invoke(prompt)
    route = response.content.strip().lower()

    return {
        "route": route
    }


def route_decision(state: AgentState) -> Literal["knowledge", "general"]:
    return state["route"]

In [ ]:
# 7. Retrieval node
def retrieve_docs_node(state: AgentState) -> dict:
    question = state["question"]

    documents = retriever.invoke(question)

    return {
        "documents": documents
    }

In [ ]:
# 8. Generate grounded RAG answer

def knowledge_answer_node(state: AgentState) -> dict:
    question = state["question"]
    documents = state["documents"]

    context = "\n\n".join(
        [
            f"Source: {doc.metadata.get('source')}\nContent: {doc.page_content}"
            for doc in documents
        ]
    )

    prompt = f"""
You are a university student services AI assistant.

Answer the user's question using ONLY the context below.

Rules:
- Do not invent information.
- If the answer is not found in the context, say:
  "I could not find this information in the university documents."
- Mention the source document when possible.
- Be clear and helpful.

User Question:
{question}

Context:
{context}
"""

    response = llm.invoke(prompt)
    return {
        "answer": response.content
    }

In [ ]:
# 9. Generate general answer
def general_answer_node(state: AgentState) -> dict:
    question = state["question"]

    prompt = f"""
Answer the user's general question clearly and briefly.

Question:
{question}
"""

    response = llm.invoke(prompt)

    return {
        "answer": response.content,
        "documents": []
    }

In [ ]:
# 10 Review answer node
def review_answer_node(state: AgentState) -> dict:
    question = state["question"]
    answer = state["answer"]
    documents = state["documents"]

    if state["route"] == "general":
        return {
            "review": "General answer. No document grounding required."
        }

    context = "\n\n".join(
        [doc.page_content for doc in documents]
    )

    prompt = f"""
You are reviewing a RAG answer.

Check whether the answer is grounded in the retrieved context.

Question:
{question}

Retrieved Context:
{context}

Generated Answer:
{answer}

Return one of the following:
- PASS: if the answer is supported by the context.
- FAIL: if the answer includes unsupported claims.
- MISSING: if the retrieved context does not contain enough information.
"""

    response = llm.invoke(prompt)

    return {
        "review": response.content
    }



In [ ]:
# 11. Finalize node

def finalize_node(state: AgentState)-> dict:
    answer = state["answer"]
    review = state["review"]

    if "FAIL" in review.upper():
        final_answer = """
I could not find enough information in the university documents to answer this question accurately.
Please contact the appropriate university office for clarification.
"""
    else:
        final_answer = answer

    return {
        "final_answer": final_answer
    }

In [ ]:
# 12. Build LangGraph workflow

builder = StateGraph(AgentState)

builder.add_node("question_router", question_router_node)
builder.add_node("retrieve_docs", retrieve_docs_node)
builder.add_node("knowledge_answer", knowledge_answer_node)
builder.add_node("general_answer", general_answer_node)
builder.add_node("review_answer", review_answer_node)
builder.add_node("finalize", finalize_node)

builder.add_edge(START, "question_router")

builder.add_conditional_edges(
    "question_router",
    route_decision,
    {
        "knowledge": "retrieve_docs",
        "general": "general_answer"
    }
)

builder.add_edge("retrieve_docs", "knowledge_answer")
builder.add_edge("knowledge_answer", "review_answer")
builder.add_edge("general_answer", "review_answer")
builder.add_edge("review_answer", "finalize")
builder.add_edge("finalize", END)

graph = builder.compile()


In [ ]:
from IPython.display import Image, display
display(Image(graph.get_graph().draw_mermaid_png()))

In [ ]:
# 13. Run test questions

def ask_agent(question:str):

    result = graph.invoke(
        {
            "question": question,
            "route": "",
            "documents": [],
            "answer": "",
            "review": "",
            "final_answer": ""
        }
    )

    print("\nQUESTION:")
    print(question)

    print("\nROUTE:")
    print(result["route"])

    print("\nANSWER:")
    print(result["final_answer"])

    print("\nREVIEW:")
    print(result["review"])

    print("\nSOURCES:")
    for doc in result["documents"]:
        print("-", doc.metadata.get("source"))

    print("\n" + "=" * 60)


In [ ]:
ask_agent("What is the GPA required to graduate from this university")